# 14. Native-600px loss sweep -- every U-Net-family arm at full resolution

Extends 05 and 13's loss-fn sweep (MAE/wavelet/starlet/gradient, 2026-09-15) across the
full pixel resolution instead of the 256px round trip every other checkpoint in this
project trains at. **This has never been done before at this scale.** `winner_native600`
(05, single arm, original loss only) has never once completed a full run (RULES.md #1) --
this notebook runs 40 arms at the same resolution, three of them (the `kin_gamma0`
arms) at 31 input channels, ~31x the tensor size of the case that already never finished.
Going in with that understood, not discovered after the fact.

**Scope, 60 arms total:**

| section | source | channels | input size | arms |
|---|---|---|---|---|
| 2 | `winner_aug_seed43`/`winner_p10_seed44`/`winner_beam_seed42`, fine-tune+fresh | 1 | 600 (true native) | 4 losses x 2 x 3 = 24 |
| 2 | core arms (`sweep_winner`, `sweep_winner_aug`, `sweep_winner_p10`, `v12_cfg`, `winner_beam`, `winner_patch`, `winner_k1`, `winner_k2`), from scratch | 1 | 600 (true native) | 8 |
| 3 | `kin_gamma0`, fine-tune+fresh | 31 | 600 (true native) | 4 losses x 2 = 8 |
| 4 | `sg_k3_fresh`, fine-tune+fresh | 7 | 600 (true native) | 4 losses x 2 = 8 |
| 5 | `ddpm_seed42` family, fine-tune+fresh | 2 (conditional) | 608 (padded, see below) | 4 losses x 2 = 8 |
| 6 | `ddrm_prior` family, fine-tune+fresh | 1 | 608 (padded, see below) | 2 losses x 2 = 4 |

**Sections 5/6 (diffusion) use `PADDED_SIZE=608`, not `NATIVE_SIZE=600` -- deliberate, not a
bug.** 600 is not a power of 2 (600 = 2^3 x 75, only 3 clean halvings before an odd number
breaks the skip-connection shapes), so a true-600px diffusion U-Net would need a 4-level
architecture (`ch_mult=[1,2,2,4]`), different from `ddpm_seed42`/`ddrm_prior`'s 5-level
`ch_mult=[1,2,2,2,4]` -- a different `ModuleList` structure, and `DenoisingDiffusion.
load_checkpoint`'s `strict=True` load would fail on every fine-tune attempt against it
(checked before writing this, not assumed).

**Fix: pad to 608 = 16 x 38 instead of changing depth.** 4 clean halvings exist
(608->304->152->76->38, verified by hand), so the SAME 5-level architecture as the
checkpoints works, with `attn_resolutions=[38]` (the bottleneck at 608px -- a different
absolute number than the checkpoint's `[16]`, but the same LEVEL INDEX, and `AttnBlock`'s
parameters depend only on channel count, not resolution, so the module structure and every
`state_dict` key/shape end up identical to the original checkpoints). 608 vs 600 is a 1.3%
upscale, not a downscale -- no native detail lost, a small price for keeping every arm's
fine-tune source usable. Sections 2-4 (fully convolutional, no depth constraint) stay at
true 600px; only the fixed-depth diffusion sections need this.

`MAX_NEW_ARMS_PER_SESSION = 1` -- the most conservative setting in the project, given zero
of this notebook's arms have ever completed once. Expect many sessions; expect some arms
(`kin_gamma0` and the `ddpm`/`ddrm` sections especially) to simply not fit in T4 memory
regardless of batch size, which is a legitimate result to report, not a bug to chase
(RULES.md's own standard: a negative result, named honestly, is a finding).


## 0. Bootstrap

In [1]:
import os, sys, subprocess, glob, re

ON_KAGGLE = os.path.exists('/kaggle')
BRANCH = 'native600-loss-sweep'
if ON_KAGGLE:
    REPO = '/kaggle/working/EXXA'; PKG = os.path.join(REPO, 'DENOISING_DIFFUSION')
    if not os.path.exists(REPO):
        subprocess.run(['git','clone','--branch',BRANCH,'--depth','1',
                        'https://github.com/KrishanYadav333/EXXA.git',REPO], check=True)
    else:
        subprocess.run(['git','-C',REPO,'fetch','origin',BRANCH], check=True)
        subprocess.run(['git','-C',REPO,'reset','--hard','origin/'+BRANCH], check=True)
    subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps',
                    'pytorch-msssim','bettermoments'], check=True)
    os.chdir(os.path.join(PKG,'notebooks')); sys.path.insert(0, PKG)

    run_re = re.compile(r'run_\d+_\d+_rt_\d+', re.I)
    roots = {}
    for p in glob.glob('/kaggle/input/**/run_*', recursive=True):
        if os.path.isdir(p) and run_re.search(os.path.basename(p)):
            roots[os.path.dirname(p)] = roots.get(os.path.dirname(p), 0) + 1
    if not roots:
        raise FileNotFoundError('No run_<id>_<step>_rt_<pp> folders under /kaggle/input. '
                                'Attach the line-emission Dataset.')
    DATA_DIR = max(roots, key=roots.get)

    sg_hits = [p for p in glob.glob('/kaggle/input/**/run_9*_rt_*', recursive=True)
              if os.path.isdir(p)]
    SG_DATA_DIR = os.path.dirname(sg_hits[0]) if sg_hits else None

    # best_models sources, uploaded under a neutral extension (RULES.md #3). Same 4
    # sources 05/13 already use: winner_aug_seed43, winner_p10_seed44, winner_beam_seed42,
    # kin_gamma0, sg_k3_fresh -- reuse whichever dataset(s) already have them attached.
    def locate_ckpt(stem):
        hits = [h for ext in ('.pth', '.ckpt', '.pth.tar')
               for h in glob.glob(f'/kaggle/input/**/{stem}{ext}', recursive=True)
               if os.path.isfile(h)]
        return hits[0] if hits else None
else:
    if os.path.basename(os.getcwd()) != 'notebooks' and os.path.isdir('notebooks'):
        os.chdir('notebooks')
    sys.path.insert(0, os.path.abspath('..'))
    DATA_DIR = '../Line Emission Data'
    SG_DATA_DIR = '../self-gravitating cube and dirty cube/sg_synth'
    def locate_ckpt(stem):
        hits = glob.glob(f'../models/best_models/{stem}.pth')
        return hits[0] if hits else None

print('DATA_DIR   :', DATA_DIR)
print('SG_DATA_DIR:', SG_DATA_DIR or 'NOT FOUND -- section 4 will be skipped')


Cloning into '/kaggle/working/EXXA'...
Updating files: 100% (5490/5490), done.


DATA_DIR   : /kaggle/input/datasets/krishanyadav333/line-emission-data/Line Emission Data
SG_DATA_DIR: /kaggle/input/datasets/krishanyadav333/exxa-sg-synth-pairs/kaggle-sg-training-dataset


## 0b. Pull latest `src` (re-run anytime -- no kernel restart needed)

In [2]:
if ON_KAGGLE:
    subprocess.run(['git','-C',REPO,'fetch','origin',BRANCH], check=True)
    subprocess.run(['git','-C',REPO,'reset','--hard','FETCH_HEAD'], check=True)
    print(subprocess.run(['git','-C',REPO,'log','--oneline','-1'], capture_output=True, text=True).stdout)
    import importlib, src; importlib.reload(src)


From https://github.com/KrishanYadav333/EXXA
 * branch            native600-loss-sweep -> FETCH_HEAD


HEAD is now at ac41eb9 Kaggle Notebook | 14-native600-loss-sweep | Version 2
ac41eb9 Kaggle Notebook | 14-native600-loss-sweep | Version 2



## 1. Imports, shared config

In [3]:
import time, csv, shutil, math
import numpy as np
import torch
from torch.utils.data import DataLoader

from src.data.cube_split import split_cubes, list_cubes
from src.data.fits_cube_dataset import FITSChannelDataset
from src.training.sweep import train_unet, LOSS_REGISTRY

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_GPU = torch.cuda.device_count()
SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

NATIVE_SIZE = 600
LOSSES = ['mae', 'wavelet', 'starlet', 'gradient']

# Nothing in this notebook has ever completed once. 1 arm/session is the most
# conservative setting anywhere in this project on purpose.
MAX_NEW_ARMS_PER_SESSION = 1
_new_arms_trained = 0
# See sweep.py / 05 / 13 for why -- fine-tune arms at full lr spike and knock out the
# pretrained weights within a few epochs.
FINETUNE_LR_SCALE = 0.1


def _cap_reached(name):
    if MAX_NEW_ARMS_PER_SESSION and _new_arms_trained >= MAX_NEW_ARMS_PER_SESSION:
        print(f'--- {name}: DEFERRED, session cap of {MAX_NEW_ARMS_PER_SESSION} new arms '
              f'reached -- resumes next session ---', flush=True)
        return True
    return False


OUT_DIR = '../results'
os.makedirs(OUT_DIR, exist_ok=True)
CKPT_DIR = '../results/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)


def persist_ckpt(path, note='', csv_path=None):
    """Copy a finished checkpoint (and its CSV) to /kaggle/working the moment it exists
    (RULES.md #1) -- CKPT_DIR/OUT_DIR are wiped by the next session's bootstrap and are
    NOT part of the notebook Output. At this notebook's per-arm cost (hours, not minutes),
    losing one to a late persist is the single most expensive mistake available here."""
    if not (ON_KAGGLE and path and os.path.exists(path)):
        return None
    base = os.path.basename(path)
    dst = os.path.join('/kaggle/working', base[:-4] if base.endswith('.pth.tar') else base)
    shutil.copy2(path, dst)
    if csv_path and os.path.exists(csv_path):
        shutil.copy2(csv_path, os.path.join('/kaggle/working', os.path.basename(csv_path)))
    print(f'    [persisted] {os.path.basename(dst)} ({os.path.getsize(dst)/1e6:.0f} MB)'
         + (f' -- {note}' if note else ''), flush=True)
    return dst


def _import_prior_nb14():
    """Restore checkpoints + CSVs from an earlier session's Output. Same reasoning as
    05's `_import_prior_nb05` / 13's `_import_prior_nb13` -- without this every session
    retrains from scratch, and at this notebook's per-arm cost that is not affordable."""
    if not ON_KAGGLE:
        return
    n_ck = 0
    for ext in ('.pth', '.ckpt', '.pth.tar'):
        for src in sorted(glob.glob(f'/kaggle/input/**/nb14_*{ext}', recursive=True)):
            if not os.path.isfile(src):
                continue
            dst = os.path.join(CKPT_DIR, os.path.basename(src)[:-len(ext)] + '.pth')
            if not os.path.exists(dst):
                shutil.copy2(src, dst)
                n_ck += 1
    n_csv = 0
    for csv_name in ('nb14_loss_sweep.csv', 'nb14_kin_loss_sweep.csv', 'nb14_sg_loss_sweep.csv'):
        dst = os.path.join(OUT_DIR, csv_name)
        if os.path.exists(dst):
            continue
        hits = glob.glob(f'/kaggle/input/**/{csv_name}', recursive=True)
        if hits:
            shutil.copy2(hits[0], dst)
            n_csv += 1
    if n_ck or n_csv:
        print(f'[nb14 prior] restored {n_ck} checkpoint(s) and {n_csv} CSV(s) from a prior Output')


def _done_rows(path, fields, ext='.pth'):
    """Rows whose CHECKPOINT is also present -- a CSV row alone is not proof an arm is
    done (RULES.md #12; caught in 13, same bug, same fix)."""
    out = {}
    if os.path.exists(path):
        with open(path, newline='') as f:
            for r in csv.DictReader(f):
                name = r['config']
                if not os.path.exists(os.path.join(CKPT_DIR, f'nb14_{name}{ext}')):
                    print(f'[resume] {name}: CSV row found but no checkpoint -- will retrain')
                    continue
                out[name] = {k: r.get(k, '') for k in fields}
    return out


_import_prior_nb14()
print(f'device: {device} | GPUs: {N_GPU} | native size: {NATIVE_SIZE} | losses: {LOSSES}')


[nb14 prior] restored 1 checkpoint(s) and 1 CSV(s) from a prior Output
device: cuda | GPUs: 2 | native size: 600 | losses: ['mae', 'wavelet', 'starlet', 'gradient']


## 2. Line-emission loss sweep at 600px -- `winner_aug_seed43` / `winner_p10_seed44` /
`winner_beam_seed42`, 4 losses x fine-tune/fresh x 3 sources = 24 arms

Same WINNER hyperparameters as every other arm in this family. `batch_size=4` -- the
convention `winner_native600` already established (05: 5.5x a 256px epoch's pixel volume).


In [4]:
WINNER = dict(base_channels=48, channel_multipliers=(1, 2, 4, 8),
              lr=8.196504330730313e-4, alpha=0.8877681051398497,
              sched_patience=8, batch_size=4)
V12_CFG = dict(base_channels=32, channel_multipliers=(1, 2, 4),
              lr=1e-3, alpha=0.8, sched_patience=5, batch_size=4)
N_SAMPLES = 100   # fewer than 05's 150 -- 600px items are already the memory pressure,
                  # no need to also inflate how many are indexed per epoch
NW = 2 if torch.cuda.is_available() else 0

train_cubes, val_cubes, _ = split_cubes(data_dir=DATA_DIR, n_holdout=3,
                                        val_fraction=0.2, seed=SEED)
_kw = dict(n_samples=N_SAMPLES, target_size=NATIVE_SIZE, seed=SEED,
          subtract_continuum=True, continuum_n=5, verbose=False)
train_ds = FITSChannelDataset(train_cubes, **_kw)
val_ds   = FITSChannelDataset(val_cubes, **_kw)
train_ds_beam = FITSChannelDataset(train_cubes, return_beam=True, **_kw)
val_ds_beam   = FITSChannelDataset(val_cubes, return_beam=True, **_kw)

# D4-augmented view, for sweep_winner_aug at 600px -- same construction as 05's own aug
# view, just at native resolution instead of 256.
train_ds_aug = FITSChannelDataset(train_cubes, augment=True, **_kw)

# Patch view -- 64px crops drawn from the 600px full images instead of 05's 256px ones.
# Same patch size, genuinely different (less-downsampled) local content.
from src.data.patches import FlatPatchDataset
PATCH_SIZE, N_PATCHES, PATCH_SIGNAL_BIAS = 64, 8, 0.5
train_ds_patch = FlatPatchDataset(train_ds, patch_size=PATCH_SIZE, n_patches=N_PATCHES,
                                  seed=SEED, signal_bias=PATCH_SIGNAL_BIAS)
val_ds_patch   = FlatPatchDataset(val_ds, patch_size=PATCH_SIZE, n_patches=N_PATCHES,
                                  seed=SEED, signal_bias=PATCH_SIGNAL_BIAS)

# 2.5D spectral-context views at 600px, matching 05's winner_k1/winner_k2.
N_NEIGHBORS_1, N_NEIGHBORS_2 = 1, 2
train_ds_k1 = FITSChannelDataset(train_cubes, n_neighbors=N_NEIGHBORS_1, **_kw)
val_ds_k1   = FITSChannelDataset(val_cubes,   n_neighbors=N_NEIGHBORS_1, **_kw)
train_ds_k2 = FITSChannelDataset(train_cubes, n_neighbors=N_NEIGHBORS_2, **_kw)
val_ds_k2   = FITSChannelDataset(val_cubes,   n_neighbors=N_NEIGHBORS_2, **_kw)

print(f'600px line-emission: train {len(train_ds)} | val {len(val_ds)}')
print(f'  aug: {len(train_ds_aug)} (8 orientations) | patch: {len(train_ds_patch)*N_PATCHES:,} | '
     f'k1: {len(train_ds_k1)} | k2: {len(train_ds_k2)}')


# Shared by both the core-arms and the loss-sweep training loops below.
FIELDS = ['config', 'source', 'psnr', 'ssim', 'mse', 'best_val_loss', 'best_epoch',
         'epochs_run', 'wall_time_s']
LOSS_CSV = os.path.join(OUT_DIR, 'nb14_loss_sweep.csv')


CUBE-LEVEL SPLIT (grouped by RunID — no channel-level leakage)
  data_dir          : /kaggle/input/datasets/krishanyadav333/line-emission-data/Line Emission Data
  total cubes       : 14  across 11 distinct RunIDs
  seed=42  n_holdout=3  val_fraction=0.2
----------------------------------------------------------------------
  TRAIN   :  7 cubes | RunIDs ['0006', '0010', '0020', '0022', '0030', '0035']
  VAL     :  2 cubes | RunIDs ['0016', '0036']
  HOLDOUT :  5 cubes | RunIDs ['0002', '0025', '0026']  <-- inference only, NEVER trained/validated
----------------------------------------------------------------------
  HOLDOUT cube folders (reserved for moment-map evaluation):
    - run_0002_00560_rt_00
    - run_0002_00560_rt_01
    - run_0002_00560_rt_04
    - run_0025_01000_rt_04
    - run_0026_00005_rt_04
600px line-emission: train 700 | val 200
  aug: 700 (8 orientations) | patch: 44,800 | k1: 700 | k2: 700


In [5]:
# ---- core arms at 600px, trained from scratch (no init_from -- these ARE the base
# arms, same as 05's own sweep_winner/sweep_winner_aug/.../winner_k2, just at native
# resolution instead of 256px). Reuses the views built above.
CORE_VIEWS = {
    'sweep_winner_600':     (WINNER,   (train_ds,       val_ds)),
    'sweep_winner_aug_600': (WINNER,   (train_ds_aug,   val_ds)),   # val never augmented
    'sweep_winner_p10_600': (dict(WINNER, patience=10), (train_ds, val_ds)),
    'v12_cfg_600':          (V12_CFG,  (train_ds,       val_ds)),
    'winner_beam_600':      (dict(WINNER, use_beam=True), (train_ds_beam, val_ds_beam)),
    # Scored on the FULL 256->native val set, not val_ds_patch -- same reasoning as 05's
    # resume branch: every model is deployed full-image, so that's the honest common
    # ground, not the arm's own training crop.
    'winner_patch_600':     (WINNER,   (train_ds_patch, val_ds)),
    'winner_k1_600':        (dict(WINNER, n_neighbors=N_NEIGHBORS_1), (train_ds_k1, val_ds_k1)),
    'winner_k2_600':        (dict(WINNER, n_neighbors=N_NEIGHBORS_2), (train_ds_k2, val_ds_k2)),
}

# Self-contained: does NOT depend on `loss_rows` from the loss-sweep cell below, since
# this cell runs FIRST (before that cell defines it). Both cells share LOSS_CSV as the
# actual source of truth; the in-memory list here is local to this cell only.
core_done = _done_rows(LOSS_CSV, FIELDS)
core_rows = list(core_done.values())
for name, (cfg, (tr, va)) in CORE_VIEWS.items():
    if name in core_done:
        print(f'--- {name}: SKIPPED, already done ---')
        continue
    if _cap_reached(name):
        continue
    ckpt = os.path.join(CKPT_DIR, f'nb14_{name}.pth')
    _cfg = dict(cfg)
    _patience = _cfg.pop('patience', 6)
    print(f'\n{"="*70}\n=== {name}\n{"="*70}', flush=True)
    res = train_unet(tr, va, device, **_cfg,
                     min_epochs=25, max_epochs=35, patience=_patience,
                     num_workers=NW, seed=SEED, ckpt_path=ckpt, verbose=True)
    row = {'config': name, 'source': 'fresh',
           **{k: res[k] for k in FIELDS if k in res}}
    core_rows.append(row)
    new = not os.path.exists(LOSS_CSV)
    with open(LOSS_CSV, 'a', newline='') as f:
        w = csv.DictWriter(f, fieldnames=FIELDS)
        if new: w.writeheader()
        w.writerow({k: row.get(k, '') for k in FIELDS})
    persist_ckpt(ckpt, name, csv_path=LOSS_CSV)
    _new_arms_trained += 1
    print(f'  {name}: PSNR {res["psnr"]:.4f} | SSIM {res["ssim"]:.4f}')
    res.pop('model', None)
    if torch.cuda.is_available(): torch.cuda.empty_cache()

print(f'\n600px core arms: {len(core_rows)} row(s)')


--- sweep_winner_600: SKIPPED, already done ---

=== sweep_winner_aug_600
  ep   1 | train 0.0142 | val 0.0050 | lr 8.2e-04 (310s) | RAM free 27.8/31.3 GB *best
  ep   2 | train 0.0035 | val 0.0032 | lr 8.2e-04 (315s) | RAM free 27.6/31.3 GB *best
  ep   3 | train 0.0069 | val 0.0336 | lr 8.2e-04 (315s) | RAM free 27.3/31.3 GB
  ep   4 | train 0.0112 | val 0.0045 | lr 8.2e-04 (314s) | RAM free 27.0/31.3 GB
  ep   5 | train 0.0034 | val 0.0031 | lr 8.2e-04 (315s) | RAM free 26.7/31.3 GB *best
  ep   6 | train 0.0026 | val 0.0028 | lr 8.2e-04 (314s) | RAM free 26.4/31.3 GB *best
  ep   7 | train 0.0027 | val 0.0024 | lr 8.2e-04 (314s) | RAM free 26.2/31.3 GB *best
  ep   8 | train 0.0026 | val 0.0024 | lr 8.2e-04 (314s) | RAM free 25.9/31.3 GB
  ep   9 | train 0.0023 | val 0.0024 | lr 8.2e-04 (313s) | RAM free 25.6/31.3 GB
  ep  10 | train 0.0022 | val 0.0024 | lr 8.2e-04 (313s) | RAM free 25.3/31.3 GB
  ep  11 | train 0.0021 | val 0.0038 | lr 8.2e-04 (314s) | RAM free 25.0/31.3 GB
  ep 

In [6]:
SOURCES = {
    'aug':  ('winner_aug_seed43',  'full'),
    'p10':  ('winner_p10_seed44',  'full'),
    'beam': ('winner_beam_seed42', 'beam'),
}
SRC_CKPT = {tag: locate_ckpt(stem) for tag, (stem, _) in SOURCES.items()}
for tag, path in SRC_CKPT.items():
    print(f'{tag} source ({SOURCES[tag][0]}):', path or 'NOT FOUND -- fine-tune arms for this source fall back to fresh init')

loss_done = _done_rows(LOSS_CSV, FIELDS)
if loss_done:
    print(f'[resume] {len(loss_done)} arm(s) already scored: {sorted(loss_done)}')

loss_rows = list(loss_done.values())
for src_tag, (src_stem, view) in SOURCES.items():
    use_beam = view == 'beam'
    tr, va = (train_ds_beam, val_ds_beam) if use_beam else (train_ds, val_ds)
    for loss_name in LOSSES:
        for source in ('finetune', 'fresh'):
            tag = 'ft' if source == 'finetune' else source
            name = f'winner_{loss_name}_{src_tag}_{tag}_600'
            if name in loss_done:
                print(f'--- {name}: SKIPPED, already done ---')
                continue
            if _cap_reached(name):
                continue
            ckpt = os.path.join(CKPT_DIR, f'nb14_{name}.pth')
            init_state = None
            if source == 'finetune':
                if SRC_CKPT[src_tag] is None:
                    print(f'--- {name}: SKIPPED, no source checkpoint for {src_tag} ---')
                    continue
                init_state = torch.load(SRC_CKPT[src_tag], map_location=device,
                                        weights_only=False)['model_state_dict']
                _min_ep = 15   # 600px epochs are ~6x a 256px epoch's wall time --
                               # 30/50 (05's budget) would be unaffordable here
                _lr = WINNER['lr'] * FINETUNE_LR_SCALE
            else:
                _min_ep = 25
                _lr = WINNER['lr']
            print(f'\n{"="*70}\n=== {name}\n{"="*70}', flush=True)
            res = train_unet(tr, va, device, **{**WINNER, 'lr': _lr, 'loss_name': loss_name,
                                                'use_beam': use_beam},
                             min_epochs=_min_ep, max_epochs=35, patience=6,
                             num_workers=NW, seed=SEED, ckpt_path=ckpt, verbose=True,
                             init_state_dict=init_state)
            row = {'config': name, 'source': source,
                   **{k: res[k] for k in FIELDS if k in res}}
            loss_rows.append(row)
            new = not os.path.exists(LOSS_CSV)
            with open(LOSS_CSV, 'a', newline='') as f:
                w = csv.DictWriter(f, fieldnames=FIELDS)
                if new: w.writeheader()
                w.writerow({k: row.get(k, '') for k in FIELDS})
            persist_ckpt(ckpt, name, csv_path=LOSS_CSV)
            _new_arms_trained += 1
            print(f'  {name}: PSNR {res["psnr"]:.4f} | SSIM {res["ssim"]:.4f}')
            res.pop('model', None)
            if torch.cuda.is_available(): torch.cuda.empty_cache()

print(f'\n600px line-emission sweep: {len(loss_rows)} row(s)')


aug source (winner_aug_seed43): /kaggle/input/datasets/krishanyadav333/exxa-sg-synth-pairs/kaggle-sg-training-dataset/winner_aug_seed43.ckpt
p10 source (winner_p10_seed44): /kaggle/input/datasets/krishanyadav333/exxa-14-checkpoint-sources/winner_p10_seed44.ckpt
beam source (winner_beam_seed42): /kaggle/input/datasets/krishanyadav333/exxa-14-checkpoint-sources/winner_beam_seed42.ckpt
[resume] 2 arm(s) already scored: ['sweep_winner_600', 'sweep_winner_aug_600']
--- winner_mae_aug_ft_600: DEFERRED, session cap of 1 new arms reached -- resumes next session ---
--- winner_mae_aug_fresh_600: DEFERRED, session cap of 1 new arms reached -- resumes next session ---
--- winner_wavelet_aug_ft_600: DEFERRED, session cap of 1 new arms reached -- resumes next session ---
--- winner_wavelet_aug_fresh_600: DEFERRED, session cap of 1 new arms reached -- resumes next session ---
--- winner_starlet_aug_ft_600: DEFERRED, session cap of 1 new arms reached -- resumes next session ---
--- winner_starlet_aug

## 3. `kin_gamma0` loss sweep at 600px -- 31 input channels, 4 losses x ft/fresh = 8 arms

**The highest-risk section in this notebook.** 31 channels x 600 x 600 is ~31x the tensor
size of `winner_native600` (1 channel), which has never completed once at batch_size=4.
`batch_size=1` here is not a tuning choice, it's the floor -- if this still does not fit in
a T4's 16 GB, that is a real answer (kin_gamma0 does not scale to native resolution on this
hardware), not a bug to keep chasing. `N_SAMPLES` and `min_epochs` both cut well below
section 2's budget for the same reason: this section is about finding out whether it runs
at all before spending real epochs on it.


In [7]:
K_KIN = 15
N_CH_KIN = 2 * K_KIN + 1
N_SAMPLES_KIN = 40

KIN_WINNER = dict(base_channels=48, channel_multipliers=(1, 2, 4, 8),
                  lr=8.196504330730313e-4, alpha=0.8877681051398497,
                  sched_patience=8, batch_size=1,
                  n_neighbors=K_KIN, out_channels=N_CH_KIN, kinematic_gamma=0.0)

train_cubes_kin, val_cubes_kin, _ = split_cubes(data_dir=DATA_DIR, n_holdout=3,
                                                val_fraction=0.2, seed=SEED)
_kw = dict(n_samples=N_SAMPLES_KIN, target_size=NATIVE_SIZE, seed=SEED,
          subtract_continuum=True, continuum_n=5,
          n_neighbors=K_KIN, stack_target=True, verbose=False)
train_ds_kin = FITSChannelDataset(train_cubes_kin, **_kw)
val_ds_kin   = FITSChannelDataset(val_cubes_kin, **_kw)
d, c = train_ds_kin[0]
assert d.shape == c.shape == (N_CH_KIN, NATIVE_SIZE, NATIVE_SIZE)
print(f'kin 600px: train {len(train_ds_kin)} | val {len(val_ds_kin)} | '
     f'{N_CH_KIN}-channel stacks | one item = {d.numel() * 4 / 1e6:.0f} MB (dirty alone)')

from astropy.io import fits
_h = fits.getheader(train_cubes_kin[0]['clean'])
VELAX_KIN = (np.arange(N_CH_KIN) - K_KIN) * float(_h['CDELT3'])


CUBE-LEVEL SPLIT (grouped by RunID — no channel-level leakage)
  data_dir          : /kaggle/input/datasets/krishanyadav333/line-emission-data/Line Emission Data
  total cubes       : 14  across 11 distinct RunIDs
  seed=42  n_holdout=3  val_fraction=0.2
----------------------------------------------------------------------
  TRAIN   :  7 cubes | RunIDs ['0006', '0010', '0020', '0022', '0030', '0035']
  VAL     :  2 cubes | RunIDs ['0016', '0036']
  HOLDOUT :  5 cubes | RunIDs ['0002', '0025', '0026']  <-- inference only, NEVER trained/validated
----------------------------------------------------------------------
  HOLDOUT cube folders (reserved for moment-map evaluation):
    - run_0002_00560_rt_00
    - run_0002_00560_rt_01
    - run_0002_00560_rt_04
    - run_0025_01000_rt_04
    - run_0026_00005_rt_04
kin 600px: train 280 | val 80 | 31-channel stacks | one item = 45 MB (dirty alone)


In [8]:
KIN_CKPT_SRC = locate_ckpt('kin_gamma0')
print('kin_gamma0 source:', KIN_CKPT_SRC or 'NOT FOUND -- fine-tune arms fall back to fresh init')

KIN_FIELDS = ['config', 'source', 'psnr', 'ssim', 'mse', 'best_val_loss', 'best_epoch',
             'epochs_run', 'wall_time_s']
KIN_CSV = os.path.join(OUT_DIR, 'nb14_kin_loss_sweep.csv')
kin_done = _done_rows(KIN_CSV, KIN_FIELDS)
if kin_done:
    print(f'[resume] {len(kin_done)} kin arm(s) already scored: {sorted(kin_done)}')

kin_rows = list(kin_done.values())
for loss_name in LOSSES:
    for source in ('finetune', 'fresh'):
        tag = 'ft' if source == 'finetune' else source
        name = f'kin_gamma0_{loss_name}_{tag}_600'
        if name in kin_done:
            print(f'--- {name}: SKIPPED, already done ---')
            continue
        if _cap_reached(name):
            continue
        ckpt = os.path.join(CKPT_DIR, f'nb14_{name}.pth')
        init_state = None
        if source == 'finetune':
            if KIN_CKPT_SRC is None:
                print(f'--- {name}: SKIPPED, no source checkpoint ---')
                continue
            init_state = torch.load(KIN_CKPT_SRC, map_location=device,
                                    weights_only=False)['model_state_dict']
            _min_ep, _lr = 8, KIN_WINNER['lr'] * FINETUNE_LR_SCALE
        else:
            _min_ep, _lr = 15, KIN_WINNER['lr']
        print(f'\n{"="*70}\n=== {name}\n{"="*70}', flush=True)
        try:
            res = train_unet(train_ds_kin, val_ds_kin, device,
                             **{**KIN_WINNER, 'lr': _lr, 'loss_name': loss_name},
                             velax_kms=VELAX_KIN, min_epochs=_min_ep, max_epochs=25, patience=5,
                             num_workers=NW, seed=SEED, ckpt_path=ckpt, verbose=True,
                             init_state_dict=init_state)
        except RuntimeError as e:
            # Catches RuntimeError, not just torch.cuda.OutOfMemoryError (a subclass only
            # in newer torch -- older versions raise a plain RuntimeError whose message
            # says 'out of memory'). A real answer, not a bug: 31ch x 600x600 may simply
            # not fit on a T4 even at batch_size=1. Record it and move on rather than let
            # one arm's OOM kill the whole session.
            if 'out of memory' not in str(e).lower():
                raise
            print(f'  [OOM] {name} does not fit in GPU memory at batch_size=1: {e}')
            torch.cuda.empty_cache()
            continue
        row = {'config': name, 'source': source,
               **{k: res[k] for k in KIN_FIELDS if k in res}}
        kin_rows.append(row)
        new = not os.path.exists(KIN_CSV)
        with open(KIN_CSV, 'a', newline='') as f:
            w = csv.DictWriter(f, fieldnames=KIN_FIELDS)
            if new: w.writeheader()
            w.writerow({k: row.get(k, '') for k in KIN_FIELDS})
        persist_ckpt(ckpt, name, csv_path=KIN_CSV)
        _new_arms_trained += 1
        print(f'  {name}: PSNR {res["psnr"]:.4f} | SSIM {res["ssim"]:.4f}')
        res.pop('model', None)
        if torch.cuda.is_available(): torch.cuda.empty_cache()

print(f'\nkin_gamma0 600px sweep: {len(kin_rows)} row(s)')


kin_gamma0 source: /kaggle/input/datasets/krishanyadav333/exxa-13-checkpoint-sources/kin_gamma0.ckpt
--- kin_gamma0_mae_ft_600: DEFERRED, session cap of 1 new arms reached -- resumes next session ---
--- kin_gamma0_mae_fresh_600: DEFERRED, session cap of 1 new arms reached -- resumes next session ---
--- kin_gamma0_wavelet_ft_600: DEFERRED, session cap of 1 new arms reached -- resumes next session ---
--- kin_gamma0_wavelet_fresh_600: DEFERRED, session cap of 1 new arms reached -- resumes next session ---
--- kin_gamma0_starlet_ft_600: DEFERRED, session cap of 1 new arms reached -- resumes next session ---
--- kin_gamma0_starlet_fresh_600: DEFERRED, session cap of 1 new arms reached -- resumes next session ---
--- kin_gamma0_gradient_ft_600: DEFERRED, session cap of 1 new arms reached -- resumes next session ---
--- kin_gamma0_gradient_fresh_600: DEFERRED, session cap of 1 new arms reached -- resumes next session ---

kin_gamma0 600px sweep: 0 row(s)


## 4. `sg_k3_fresh` loss sweep at 600px -- 7 input channels, 4 losses x ft/fresh = 8 arms

SG cubes are also native 601x600x600 (`results/self-gravitating/README.md`), so this is the
same mechanical change as section 2 -- 7 channels is much closer to section 2's 1-channel
case than to section 3's 31-channel one. `batch_size=2`, between section 2's 4 and section
3's 1.


In [9]:
if SG_DATA_DIR is None:
    print('SG_DATA_DIR not found -- section 4 skipped.')
else:
    K_SG = 3
    N_SAMPLES_SG = 60

    SG_WINNER = dict(base_channels=48, channel_multipliers=(1, 2, 4, 8),
                     lr=8.196504330730313e-4, alpha=0.8877681051398497,
                     sched_patience=8, batch_size=2,
                     n_neighbors=K_SG, out_channels=1)

    TRAIN_RUNS_SG = ['run_9015_00370_rt_00', 'run_9019_00019_rt_00', 'run_9032_00020_rt_00']
    VAL_RUN_SG = 'run_9025_00370_rt_00'

    all_cubes_sg = {c['folder']: c for c in list_cubes(SG_DATA_DIR)}
    train_cubes_sg = [all_cubes_sg[r] for r in TRAIN_RUNS_SG]
    val_cubes_sg = [all_cubes_sg[VAL_RUN_SG]]

    _kw = dict(n_samples=N_SAMPLES_SG, target_size=NATIVE_SIZE, seed=SEED,
              subtract_continuum=False, n_neighbors=K_SG, stack_target=False, verbose=False)
    train_ds_sg = FITSChannelDataset(train_cubes_sg, **_kw)
    val_ds_sg   = FITSChannelDataset(val_cubes_sg, **_kw)
    d, c = train_ds_sg[0]
    assert d.shape[0] == 2 * K_SG + 1 and c.shape[0] == 1
    print(f'sg 600px: train {len(train_ds_sg)} | val {len(val_ds_sg)} | '
         f'{2*K_SG+1}-channel input, 1-ch target')


sg 600px: train 180 | val 60 | 7-channel input, 1-ch target


In [10]:
if SG_DATA_DIR is None:
    print('section 4 skipped')
else:
    SG_CKPT_SRC = locate_ckpt('sg_k3_fresh')
    print('sg_k3_fresh source:', SG_CKPT_SRC or 'NOT FOUND -- fine-tune arms fall back to fresh init')

    SG_FIELDS = ['config', 'source', 'psnr', 'ssim', 'mse', 'best_val_loss', 'best_epoch',
                'epochs_run', 'wall_time_s']
    SG_CSV = os.path.join(OUT_DIR, 'nb14_sg_loss_sweep.csv')
    sg_done = _done_rows(SG_CSV, SG_FIELDS)
    if sg_done:
        print(f'[resume] {len(sg_done)} sg arm(s) already scored: {sorted(sg_done)}')

    sg_rows = list(sg_done.values())
    for loss_name in LOSSES:
        for source in ('finetune', 'fresh'):
            tag = 'ft' if source == 'finetune' else source
            name = f'sg_k3_{loss_name}_{tag}_600'
            if name in sg_done:
                print(f'--- {name}: SKIPPED, already done ---')
                continue
            if _cap_reached(name):
                continue
            ckpt = os.path.join(CKPT_DIR, f'nb14_{name}.pth')
            init_state = None
            if source == 'finetune':
                if SG_CKPT_SRC is None:
                    print(f'--- {name}: SKIPPED, no source checkpoint ---')
                    continue
                init_state = torch.load(SG_CKPT_SRC, map_location=device,
                                        weights_only=False)['model_state_dict']
                _min_ep, _lr = 10, SG_WINNER['lr'] * FINETUNE_LR_SCALE
            else:
                _min_ep, _lr = 20, SG_WINNER['lr']
            print(f'\n{"="*70}\n=== {name}\n{"="*70}', flush=True)
            try:
                res = train_unet(train_ds_sg, val_ds_sg, device,
                                 **{**SG_WINNER, 'lr': _lr, 'loss_name': loss_name},
                                 min_epochs=_min_ep, max_epochs=35, patience=6,
                                 num_workers=NW, seed=SEED, ckpt_path=ckpt, verbose=True,
                                 init_state_dict=init_state)
            except RuntimeError as e:
                if 'out of memory' not in str(e).lower():
                    raise
                print(f'  [OOM] {name} does not fit in GPU memory at batch_size=2: {e}')
                torch.cuda.empty_cache()
                continue
            row = {'config': name, 'source': source,
                   **{k: res[k] for k in SG_FIELDS if k in res}}
            sg_rows.append(row)
            new = not os.path.exists(SG_CSV)
            with open(SG_CSV, 'a', newline='') as f:
                w = csv.DictWriter(f, fieldnames=SG_FIELDS)
                if new: w.writeheader()
                w.writerow({k: row.get(k, '') for k in SG_FIELDS})
            persist_ckpt(ckpt, name, csv_path=SG_CSV)
            _new_arms_trained += 1
            print(f'  {name}: PSNR {res["psnr"]:.4f} | SSIM {res["ssim"]:.4f}')
            res.pop('model', None)
            if torch.cuda.is_available(): torch.cuda.empty_cache()

    print(f'\nsg_k3_fresh 600px sweep: {len(sg_rows)} row(s)')


sg_k3_fresh source: /kaggle/input/datasets/krishanyadav333/exxa-13-checkpoint-sources/sg_k3_fresh.ckpt
--- sg_k3_mae_ft_600: DEFERRED, session cap of 1 new arms reached -- resumes next session ---
--- sg_k3_mae_fresh_600: DEFERRED, session cap of 1 new arms reached -- resumes next session ---
--- sg_k3_wavelet_ft_600: DEFERRED, session cap of 1 new arms reached -- resumes next session ---
--- sg_k3_wavelet_fresh_600: DEFERRED, session cap of 1 new arms reached -- resumes next session ---
--- sg_k3_starlet_ft_600: DEFERRED, session cap of 1 new arms reached -- resumes next session ---
--- sg_k3_starlet_fresh_600: DEFERRED, session cap of 1 new arms reached -- resumes next session ---
--- sg_k3_gradient_ft_600: DEFERRED, session cap of 1 new arms reached -- resumes next session ---
--- sg_k3_gradient_fresh_600: DEFERRED, session cap of 1 new arms reached -- resumes next session ---

sg_k3_fresh 600px sweep: 0 row(s)


## 5. Diffusion -- `ddpm_seed42` loss sweep at (padded) native resolution

**Fine-tuning restored -- the depth mismatch from the first version of this section is
fixed by padding, not by giving up on fine-tuning.** 600 is not a power of 2 (600 = 2^3 x
75), so a native 600px input only supports 3 clean halvings before the 4th hits an odd
number and breaks the skip-connection shapes -- that forced a 4-level architecture,
different from `ddpm_seed42`'s 5-level `ch_mult=[1,2,2,2,4]`, and `DenoisingDiffusion.
load_checkpoint`'s `strict=True` load would fail on every attempt against a 4-level model
(different number of `down`/`up` `ModuleList` entries -- verified before writing the first
version of this section, not assumed).

**Fix: pad to `PADDED_SIZE = 608` instead of changing depth.** 608 = 16 x 38, so 4 clean
halvings exist (608 -> 304 -> 152 -> 76 -> 38, checked by hand, every division lands on an
even number until the last): `ch_mult=[1,2,2,2,4]`, the SAME 5 levels as the checkpoint.
`attn_resolutions=[38]` instead of the checkpoint's `[16]` -- a different absolute number,
but attention fires at the same LEVEL INDEX (the bottleneck, level 4 of 5) in both
configs, and `AttnBlock`'s parameters depend only on channel count, not spatial size --
so the resulting module structure and every `state_dict` key/shape is IDENTICAL to
`ddpm_seed42`. `strict=True` loading now succeeds.

**Cost of this choice: 608 vs the cube's true 600 is a 1.3% UPSCALE, not a downscale --**
you lose no native detail, you add a negligible margin. A small price for keeping every
arm's fine-tune source usable, and a tiny fraction of the 600->256->600 round trip this
whole sweep exists to avoid. Sections 2-4 (fully convolutional, no depth constraint) stay
at true 600px -- this padding is specific to sections 5/6's fixed-depth diffusion U-Net.

Same `loss_type`/`aux_loss_name` mechanism as 13. **`AUX_WEIGHT` rescaled for 608px**: the
primary loss sums squared error over every pixel (~369,664 terms at 608px vs ~65,536 at
256px, a 5.64x ratio) while the aux term is a per-pixel mean -- 13's `AUX_WEIGHT=2000`
carried over unscaled would make the aux term relatively that much weaker than intended.
Scaled to ~11280 -- still an order-of-magnitude estimate, not tuned.

`batch_size=1`, DDIM evaluation cost at this resolution is unmeasured anywhere in this
project -- `sampling_timesteps`/`n_avg` both cut below 13's 256px values to bound the
unknown cost, a named trade-off. Wrapped in the same `OutOfMemory` catch as section 3.


In [11]:
from src.data.stacked_pair import StackedPairDataset
from src.models.diffusion_unet import default_diffusion_config
from src.training.diffusion import DenoisingDiffusion

PADDED_SIZE = 608   # 16 x 38 -- see markdown: nearest size >= 600 giving 4 clean halvings
                    # for the checkpoint's 5-level architecture. NOT the same as NATIVE_SIZE
                    # (600, used by sections 2-4) -- this is specific to the fixed-depth
                    # diffusion U-Net.
BATCH_SIZE_DDPM = 1
LR_DDPM = 2e-4
AUX_WEIGHT_608 = 2000.0 * (PADDED_SIZE ** 2) / (256 ** 2)   # ~11280 -- see markdown above
SAMPLING_STEPS_608 = 15    # 13's 256px value: 25 -- cut given unmeasured cost at this size
K_AVG_608 = 2                # 13's 256px value: 4

train_cubes_ddpm, val_cubes_ddpm, _ = split_cubes(data_dir=DATA_DIR, n_holdout=3,
                                                   val_fraction=0.2, seed=SEED)
_kw = dict(n_samples=40, target_size=PADDED_SIZE, seed=SEED,
          subtract_continuum=True, continuum_n=5, verbose=False)
_train_pairs_ddpm = FITSChannelDataset(train_cubes_ddpm, **_kw)
_val_pairs_ddpm   = FITSChannelDataset(val_cubes_ddpm, **_kw)
ddpm_train_ds = StackedPairDataset(_train_pairs_ddpm)
ddpm_val_ds   = StackedPairDataset(_val_pairs_ddpm)
_nw = 2 if ON_KAGGLE else 0
ddpm_train_loader = DataLoader(ddpm_train_ds, batch_size=BATCH_SIZE_DDPM, shuffle=True,
                               num_workers=_nw, pin_memory=True,
                               persistent_workers=_nw > 0)
ddpm_val_loader = DataLoader(ddpm_val_ds, batch_size=BATCH_SIZE_DDPM, shuffle=False,
                             num_workers=_nw, pin_memory=True,
                             persistent_workers=_nw > 0)
print(f'ddpm {PADDED_SIZE}px (padded from {NATIVE_SIZE} native): train {len(ddpm_train_ds)} | '
     f'val {len(ddpm_val_ds)} items')


def make_ddpm_cfg_608(loss_type='l2', aux_loss_name=None, aux_weight=0.0):
    c = default_diffusion_config(image_size=PADDED_SIZE)
    c.model.ch_mult = [1, 2, 2, 2, 4]      # SAME 5 levels as ddpm_seed42 -- see markdown
    c.model.attn_resolutions = [38]        # bottleneck at 608px; must be set explicitly
    c.model.ema_rate = 0.99
    c.diffusion.prediction_type = 'v'
    c.diffusion.beta_schedule = 'cosine'
    c.diffusion.min_snr_gamma = 0.0
    c.diffusion.loss_type = loss_type
    c.diffusion.aux_loss_name = aux_loss_name
    c.diffusion.aux_weight = aux_weight
    return c


CUBE-LEVEL SPLIT (grouped by RunID — no channel-level leakage)
  data_dir          : /kaggle/input/datasets/krishanyadav333/line-emission-data/Line Emission Data
  total cubes       : 14  across 11 distinct RunIDs
  seed=42  n_holdout=3  val_fraction=0.2
----------------------------------------------------------------------
  TRAIN   :  7 cubes | RunIDs ['0006', '0010', '0020', '0022', '0030', '0035']
  VAL     :  2 cubes | RunIDs ['0016', '0036']
  HOLDOUT :  5 cubes | RunIDs ['0002', '0025', '0026']  <-- inference only, NEVER trained/validated
----------------------------------------------------------------------
  HOLDOUT cube folders (reserved for moment-map evaluation):
    - run_0002_00560_rt_00
    - run_0002_00560_rt_01
    - run_0002_00560_rt_04
    - run_0025_01000_rt_04
    - run_0026_00005_rt_04
ddpm 608px (padded from 600 native): train 280 | val 80 items


In [12]:
DDPM_CKPT_SRC = locate_ckpt('ddpm_seed42')
print('ddpm_seed42 source:', DDPM_CKPT_SRC or 'NOT FOUND -- fine-tune arms fall back to fresh init')

DDPM_ARMS = ([('l1', None, 0.0)] +
            [('l2', name, AUX_WEIGHT_608) for name in ('wavelet', 'starlet', 'gradient')])

DDPM_FIELDS = ['config', 'source', 'loss_type', 'aux_loss_name', 'psnr', 'ssim', 'mse',
              'best_val_loss', 'wall_time_s']
DDPM_CSV = os.path.join(OUT_DIR, 'nb14_ddpm_loss_sweep.csv')
ddpm_done = _done_rows(DDPM_CSV, DDPM_FIELDS)
if ddpm_done:
    print(f'[resume] {len(ddpm_done)} ddpm arm(s) already scored: {sorted(ddpm_done)}')

ddpm_rows = list(ddpm_done.values())
for loss_type, aux_name, aux_w in DDPM_ARMS:
    tag = aux_name or loss_type
    for source in ('finetune', 'fresh'):
        stag = 'ft' if source == 'finetune' else source
        name = f'ddpm_{tag}_{stag}_608'
        if name in ddpm_done:
            print(f'--- {name}: SKIPPED, already done ---')
            continue
        if source == 'finetune' and DDPM_CKPT_SRC is None:
            print(f'--- {name}: SKIPPED, no source checkpoint to fine-tune from ---')
            continue
        if _cap_reached(name):
            continue
        ckpt = os.path.join(CKPT_DIR, f'nb14_{name}.pth.tar')
        print(f'\n{"="*70}\n=== {name}\n{"="*70}', flush=True)
        t0 = time.time()
        try:
            d = DenoisingDiffusion(config=make_ddpm_cfg_608(loss_type, aux_name, aux_w),
                                   device=str(device),
                                   lr=LR_DDPM * (FINETUNE_LR_SCALE if source == 'finetune' else 1.0),
                                   checkpoint_path=ckpt,
                                   data_parallel=False)
            n_epochs = 8
            if source == 'finetune':
                d.load_checkpoint(DDPM_CKPT_SRC)   # strict=True -- succeeds because the
                                                    # module structure matches exactly
                                                    # (same 5 levels, attn at the same level
                                                    # index); see markdown for why this is
                                                    # true and not assumed.
                d.best_val_loss = float('inf')   # RULES.md #4 -- old objective's scale
                                                  # doesn't carry over to the new loss
            else:
                n_epochs = 15
            d.train(ddpm_train_loader, ddpm_val_loader, n_epochs=n_epochs, verbose=True)
            m = d.evaluate(ddpm_val_loader, sampling_timesteps=SAMPLING_STEPS_608,
                           use_ema=True, n_avg=K_AVG_608)
        except RuntimeError as e:
            if 'out of memory' not in str(e).lower():
                raise
            print(f'  [OOM] {name} does not fit in GPU memory at batch_size={BATCH_SIZE_DDPM}: {e}')
            torch.cuda.empty_cache()
            continue
        row = {'config': name, 'source': source, 'loss_type': loss_type,
              'aux_loss_name': aux_name or '', 'psnr': round(float(m['psnr']), 4),
              'ssim': round(float(m['ssim']), 5), 'mse': round(float(m['mse']), 8),
              'best_val_loss': round(float(d.best_val_loss), 5),
              'wall_time_s': round(time.time() - t0, 1)}
        ddpm_rows.append(row)
        new = not os.path.exists(DDPM_CSV)
        with open(DDPM_CSV, 'a', newline='') as f:
            w = csv.DictWriter(f, fieldnames=DDPM_FIELDS)
            if new: w.writeheader()
            w.writerow({k: row.get(k, '') for k in DDPM_FIELDS})
        persist_ckpt(ckpt, name, csv_path=DDPM_CSV)
        _new_arms_trained += 1
        print(f'  {name}: PSNR {row["psnr"]:.4f} | SSIM {row["ssim"]:.4f}')
        del d
        if torch.cuda.is_available(): torch.cuda.empty_cache()

print(f'\nddpm_seed42 {PADDED_SIZE}px sweep: {len(ddpm_rows)} row(s)')


ddpm_seed42 source: /kaggle/input/datasets/krishanyadav333/exxa-13-checkpoint-sources/ddpm_seed42.ckpt
--- ddpm_l1_ft_608: DEFERRED, session cap of 1 new arms reached -- resumes next session ---
--- ddpm_l1_fresh_608: DEFERRED, session cap of 1 new arms reached -- resumes next session ---
--- ddpm_wavelet_ft_608: DEFERRED, session cap of 1 new arms reached -- resumes next session ---
--- ddpm_wavelet_fresh_608: DEFERRED, session cap of 1 new arms reached -- resumes next session ---
--- ddpm_starlet_ft_608: DEFERRED, session cap of 1 new arms reached -- resumes next session ---
--- ddpm_starlet_fresh_608: DEFERRED, session cap of 1 new arms reached -- resumes next session ---
--- ddpm_gradient_ft_608: DEFERRED, session cap of 1 new arms reached -- resumes next session ---
--- ddpm_gradient_fresh_608: DEFERRED, session cap of 1 new arms reached -- resumes next session ---

ddpm_seed42 608px sweep: 0 row(s)


## 6. Diffusion -- `ddrm_prior` loss sweep at (padded) native resolution

Same fix as section 5 (`PADDED_SIZE=608`, `ch_mult=[1,2,2,2,4]`, `attn_resolutions=[38]`,
matching `ddrm_prior`'s 5-level architecture exactly) -- fine-tuning restored. `conditional
=False` (unconditional prior, `CleanOnly` wrapper, 1 input channel not 2), so lighter than
section 5: `batch_size=2` rather than 1.

This is currently `models/best_models/README.md`'s explicit negative result -- worst of 4
methods in every wiggle confirmation to date at 256px. A meaningful outcome here is either
arm helping enough to reopen the question, not necessarily beating the U-Net arms.


In [13]:
if SG_DATA_DIR is None:
    print('SG_DATA_DIR not found -- section 6 skipped.')
else:
    BATCH_SIZE_DDRM = 2
    LR_DDRM = 2e-5

    train_cubes_ddrm, val_cubes_ddrm, _ = split_cubes(data_dir=DATA_DIR, n_holdout=3,
                                                       val_fraction=0.2, seed=SEED)
    _kw = dict(n_samples=60, target_size=PADDED_SIZE, seed=SEED,
              subtract_continuum=True, continuum_n=5, verbose=False)
    _train_pairs_ddrm = FITSChannelDataset(train_cubes_ddrm, **_kw)
    _val_pairs_ddrm   = FITSChannelDataset(val_cubes_ddrm, **_kw)

    class CleanOnly(torch.utils.data.Dataset):
        """(dirty, clean) -> clean. The prior never sees a dirty image."""
        def __init__(self, pairs): self.pairs = pairs
        def __len__(self): return len(self.pairs)
        def __getitem__(self, i): return self.pairs[i][1]

    ddrm_train_ds, ddrm_val_ds = CleanOnly(_train_pairs_ddrm), CleanOnly(_val_pairs_ddrm)
    _nw = 2 if ON_KAGGLE else 0
    ddrm_train_loader = DataLoader(ddrm_train_ds, batch_size=BATCH_SIZE_DDRM, shuffle=True,
                                   num_workers=_nw, pin_memory=True, drop_last=True,
                                   persistent_workers=_nw > 0)
    ddrm_val_loader = DataLoader(ddrm_val_ds, batch_size=BATCH_SIZE_DDRM, shuffle=False,
                                 num_workers=_nw, persistent_workers=_nw > 0)
    print(f'ddrm {PADDED_SIZE}px: train {len(ddrm_train_ds)} | val {len(ddrm_val_ds)} images')

    def make_ddrm_cfg_608(loss_type='l2', aux_loss_name=None, aux_weight=0.0):
        c = default_diffusion_config(image_size=PADDED_SIZE)
        c.model.ch_mult = [1, 2, 2, 2, 4]     # SAME 5 levels as ddrm_prior
        c.model.attn_resolutions = [38]
        c.data.conditional = False
        c.diffusion.prediction_type = 'v'
        c.diffusion.beta_schedule = 'cosine'
        c.diffusion.min_snr_gamma = 5.0
        c.diffusion.loss_type = loss_type
        c.diffusion.aux_loss_name = aux_loss_name
        c.diffusion.aux_weight = aux_weight
        return c


CUBE-LEVEL SPLIT (grouped by RunID — no channel-level leakage)
  data_dir          : /kaggle/input/datasets/krishanyadav333/line-emission-data/Line Emission Data
  total cubes       : 14  across 11 distinct RunIDs
  seed=42  n_holdout=3  val_fraction=0.2
----------------------------------------------------------------------
  TRAIN   :  7 cubes | RunIDs ['0006', '0010', '0020', '0022', '0030', '0035']
  VAL     :  2 cubes | RunIDs ['0016', '0036']
  HOLDOUT :  5 cubes | RunIDs ['0002', '0025', '0026']  <-- inference only, NEVER trained/validated
----------------------------------------------------------------------
  HOLDOUT cube folders (reserved for moment-map evaluation):
    - run_0002_00560_rt_00
    - run_0002_00560_rt_01
    - run_0002_00560_rt_04
    - run_0025_01000_rt_04
    - run_0026_00005_rt_04
ddrm 608px: train 420 | val 120 images


In [14]:
if SG_DATA_DIR is None:
    print('section 6 skipped')
else:
    DDRM_CKPT_SRC = locate_ckpt('ddrm_prior')
    print('ddrm_prior source:', DDRM_CKPT_SRC or 'NOT FOUND -- fine-tune arms fall back to fresh init')

    # No aux term: unconditional, no 'clean' to compare x0_hat against (needs x0.shape[1]>1).
    DDRM_ARMS = [('l1', None, 0.0), ('l2', None, 0.0)]

    DDRM_FIELDS = ['config', 'source', 'loss_type', 'best_val_loss', 'wall_time_s']
    DDRM_CSV = os.path.join(OUT_DIR, 'nb14_ddrm_loss_sweep.csv')
    ddrm_done = _done_rows(DDRM_CSV, DDRM_FIELDS)
    if ddrm_done:
        print(f'[resume] {len(ddrm_done)} ddrm arm(s) already scored: {sorted(ddrm_done)}')

    ddrm_rows = list(ddrm_done.values())
    for loss_type, aux_name, aux_w in DDRM_ARMS:
        for source in ('finetune', 'fresh'):
            stag = 'ft' if source == 'finetune' else source
            name = f'ddrm_{loss_type}_{stag}_608'
            if name in ddrm_done:
                print(f'--- {name}: SKIPPED, already done ---')
                continue
            if source == 'finetune' and DDRM_CKPT_SRC is None:
                print(f'--- {name}: SKIPPED, no source checkpoint to fine-tune from ---')
                continue
            if _cap_reached(name):
                continue
            ckpt = os.path.join(CKPT_DIR, f'nb14_{name}.pth.tar')
            print(f'\n{"="*70}\n=== {name}\n{"="*70}', flush=True)
            t0 = time.time()
            try:
                d = DenoisingDiffusion(config=make_ddrm_cfg_608(loss_type, aux_name, aux_w),
                                       device=str(device),
                                       lr=LR_DDRM * (FINETUNE_LR_SCALE if source == 'finetune' else 1.0),
                                       checkpoint_path=ckpt, data_parallel=False)
                n_epochs = 8
                if source == 'finetune':
                    d.load_checkpoint(DDRM_CKPT_SRC)
                    d.best_val_loss = float('inf')
                else:
                    n_epochs = 15
                d.train(ddrm_train_loader, ddrm_val_loader, n_epochs=n_epochs, verbose=True)
            except RuntimeError as e:
                if 'out of memory' not in str(e).lower():
                    raise
                print(f'  [OOM] {name} does not fit in GPU memory at batch_size={BATCH_SIZE_DDRM}: {e}')
                torch.cuda.empty_cache()
                continue
            row = {'config': name, 'source': source, 'loss_type': loss_type,
                  'best_val_loss': round(float(d.best_val_loss), 5),
                  'wall_time_s': round(time.time() - t0, 1)}
            ddrm_rows.append(row)
            new = not os.path.exists(DDRM_CSV)
            with open(DDRM_CSV, 'a', newline='') as f:
                w = csv.DictWriter(f, fieldnames=DDRM_FIELDS)
                if new: w.writeheader()
                w.writerow({k: row.get(k, '') for k in DDRM_FIELDS})
            persist_ckpt(ckpt, name, csv_path=DDRM_CSV)
            _new_arms_trained += 1
            print(f'  {name}: best_val_loss {row["best_val_loss"]:.5f}')
            del d
            if torch.cuda.is_available(): torch.cuda.empty_cache()

    print(f'\nddrm_prior {PADDED_SIZE}px sweep: {len(ddrm_rows)} row(s)')


ddrm_prior source: /kaggle/input/datasets/krishanyadav333/exxa-13-checkpoint-sources/ddrm_prior.ckpt
--- ddrm_l1_ft_608: DEFERRED, session cap of 1 new arms reached -- resumes next session ---
--- ddrm_l1_fresh_608: DEFERRED, session cap of 1 new arms reached -- resumes next session ---
--- ddrm_l2_ft_608: DEFERRED, session cap of 1 new arms reached -- resumes next session ---
--- ddrm_l2_fresh_608: DEFERRED, session cap of 1 new arms reached -- resumes next session ---

ddrm_prior 608px sweep: 0 row(s)


## 7. Collect outputs

In [15]:
from src.evaluation.collect_outputs import collect_outputs

_run_dir = collect_outputs(
    '14-native600-loss-sweep',
    [
        'nb14_loss_sweep.csv',
        'nb14_kin_loss_sweep.csv',
        'nb14_sg_loss_sweep.csv',
        'nb14_ddpm_loss_sweep.csv',
        'nb14_ddrm_loss_sweep.csv',
    ],
    extra={'losses': LOSSES, 'native_size': NATIVE_SIZE,
          'sg_ckpt_found': SG_DATA_DIR is not None and 'SG_CKPT_SRC' in dir() and SG_CKPT_SRC is not None},
)
print('\nAnything listed as NOT FOUND above did not get written this run.')


collected 1 file(s), 0.0 MiB -> /kaggle/working/outputs/14-native600-loss-sweep/2026-09-24T153049_ac41eb9
       0.00 MiB  nb14_loss_sweep.csv
   NOT FOUND (not written this run?): ['nb14_kin_loss_sweep.csv', 'nb14_sg_loss_sweep.csv', 'nb14_ddpm_loss_sweep.csv', 'nb14_ddrm_loss_sweep.csv']

Anything listed as NOT FOUND above did not get written this run.
